# Convert a COSI point-source response for XSPEC

This notebook converts a COSI full detector response into target- and observation-specific OGIP files that XSPEC can read:

- an ancillary response file (ARF) containing the effective area; and
- a redistribution matrix file (RMF) containing the incident-to-measured energy redistribution.

The conversion projects the full Compton data-space response onto energy. It therefore preserves the energy response, but discards the directional and Compton-scattering information used by the native COSI likelihood. This is appropriate for an energy-only XSPEC analysis or a joint XMM--Newton, NuSTAR, and COSI demonstration.

For an XSPEC simulation, the recommended workflow is to create the ARF and RMF here and then use XSPEC `fakeit` with the desired spectral model.

## 1. Imports

In [ ]:
from pathlib import Path

import numpy as np
import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.io import fits

from cosipy import SpacecraftHistory
from cosipy.event_selection import GoodTimeInterval
from cosipy.response import FullDetectorResponse, RspArfRmfConverter

## 2. Configure the input and output paths

This simulation-only workflow uses the COSI response and orientation from `NGC4151_Paper_Plots_TH_Joint_XMM.ipynb`, plus the generic XMM--Newton and NuSTAR responses under `XRay/Simulations`. No observed XMM or NuSTAR source spectra are required. The COSI ARF depends on the response, pointing history, live time, target position, and field-of-view selection, so the 60-degree pointing cut is reproduced below.

In [ ]:
analysis_root = Path(
    "/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/"
    "COSI/Radio_Quiet_AGN"
)
dc4_dir = analysis_root / "DC4_Files"
xray_root = analysis_root / "XRay"

response_path = dc4_dir / (
    "ResponseContinuum.o3.e100_10000.b10log.s10396905069491."
    "m2284.filtered.nonsparse.binnedimaging.imagingresponse.h5"
)
orientation_path = dc4_dir / (
    "DC4_final_530km_3_month_with_slew_15sbins_GalacticEarth_SAA.fits"
)

# Generic response products for creating forecast spectra with XSPEC fakeit.
simulation_dir = xray_root / "Simulations"
xmm_sim_dir = simulation_dir / "XMM"
xmm_sim_rmf = xmm_sim_dir / "pn-thin-5-ao26.rmf"
xmm_sim_arf = xmm_sim_dir / "pn-thin-5-ao26.arf"
nustar_sim_dir = simulation_dir / "NuSTAR/point_sources_for_proposers_v3"
nustar_sim_rmf = nustar_sim_dir / "nustar.rmf"
nustar_sim_arf = nustar_sim_dir / "point_60arcsecRad_1arcminOA.arf"

output_dir = simulation_dir / "COSI"
output_dir.mkdir(parents=True, exist_ok=True)

target_coord = SkyCoord(l=155.077, b=75.063, frame="galactic", unit="deg")
fov_cut = 60 * u.deg

if not response_path.exists():
    raise FileNotFoundError(f"Set response_path to an existing file: {response_path}")
if not orientation_path.exists():
    raise FileNotFoundError(f"Set orientation_path to an existing file: {orientation_path}")

simulation_inputs = {
    "XMM simulation RMF": xmm_sim_rmf,
    "XMM simulation ARF": xmm_sim_arf,
    "NuSTAR simulation RMF": nustar_sim_rmf,
    "NuSTAR simulation ARF": nustar_sim_arf,
}
for label, path in simulation_inputs.items():
    status = "found" if path.exists() else "MISSING"
    print(f"{status:7s}  {label}: {path}")

## 3. Build the point-source spectral response

`RspArfRmfConverter` uses the spacecraft history to form the dwell map for the target and then projects the point-source response onto incident energy (`Ei`) and measured energy (`Em`).

In [ ]:
response = FullDetectorResponse.open(response_path)
full_orientation = SpacecraftHistory.open(orientation_path)
source_gti = GoodTimeInterval.from_pointing_cut(
    target_coord,
    full_orientation,
    fov_cut,
    earth_occ=False,
)
orientation = full_orientation.apply_gti(source_gti)

converter = RspArfRmfConverter(
    response=response,
    ori=orientation,
    target_coord=target_coord,
)

(
    Ei_edges, Ei_lo, Ei_hi,
    Em_edges, Em_lo, Em_hi,
    effective_area, redistribution,
) = converter.get_psr_rsp()

exposure_s = float(orientation.livetime.to_value(u.s).sum())
print(f"Exposure: {exposure_s:.3f} s")
print(f"Incident-energy bins: {len(Ei_lo)}")
print(f"Measured-energy channels: {len(Em_lo)}")

## 4. Validate the projected response

The present converter has been tested with matching `Ei` and `Em` grids. It uses the measured-energy edges in portions of the ARF/RMF writer, so stop here if the grids differ. The converter should be generalized before exporting such a response. Also check for zero-area incident bins, which can create undefined rows when the redistribution is normalized.

In [ ]:
same_energy_grid = (
    Ei_edges.shape == Em_edges.shape
    and np.allclose(Ei_edges, Em_edges, rtol=1e-6, atol=0.0)
)

checks = {
    "matching Ei/Em grids": same_energy_grid,
    "finite effective area": np.all(np.isfinite(effective_area)),
    "finite redistribution": np.all(np.isfinite(redistribution)),
    "positive exposure": exposure_s > 0.0,
}

for name, passed in checks.items():
    print(f"{name:30s}: {passed}")

if not same_energy_grid:
    raise RuntimeError(
        "The current OGIP writer assumes matching Ei and Em grids. "
        "Do not export this response until the writer is generalized."
    )

if not checks["finite effective area"] or not checks["finite redistribution"]:
    raise RuntimeError("The projected response contains NaN or infinite values.")

zero_area = np.flatnonzero(effective_area <= 0.0)
if zero_area.size:
    print("Warning: zero-area incident bins:", zero_area.tolist())

## 5. Write the XSPEC ARF and RMF

In [ ]:
arf_path = output_dir / "cosi_ngc4151.arf"
rmf_path = output_dir / "cosi_ngc4151.rmf"

converter.write_arf(arf_path, overwrite=True)
converter.write_rmf(rmf_path, overwrite=True)

print(arf_path.resolve())
print(rmf_path.resolve())

In [ ]:
converter.plot_arf()
converter.plot_rmf();

## 6. Inspect the OGIP files

This is a lightweight structural check. If HEASoft is active, also run `ftverify` on both files from a terminal.

In [ ]:
with fits.open(arf_path) as hdul:
    hdul.info()
    print("ARF rows:", len(hdul["SPECRESP"].data))
    print("ARF energy range [keV]:",
          hdul["SPECRESP"].data["ENERG_LO"][0],
          hdul["SPECRESP"].data["ENERG_HI"][-1])

with fits.open(rmf_path) as hdul:
    hdul.info()
    print("RMF incident bins:", len(hdul["MATRIX"].data))
    print("RMF channels:", hdul["MATRIX"].header["DETCHANS"])

print("Run these in a HEASoft terminal:")
print(f"ftverify {arf_path}")
print(f"ftverify {rmf_path}")

## 7A. Recommended simulation: use XSPEC `fakeit`

For an XSPEC-only energy analysis, generate the ARF/RMF above and let XSPEC fold the desired RELXILL model through them. This avoids translating RELXILL into an astromodels function.

A minimal XSPEC session is:

```xspec
lmod relxill /Users/parshadkp/Software/XSPEC_models/relxill
abund wilm
xsect vern
model constant*TBabs*((partcov*zTBabs)*(xillver+highecut*zpowerlw+relconv*xillver))
fakeit
```

At the COSI `fakeit` prompts, provide `cosi_ngc4151.rmf`, `cosi_ngc4151.arf`, and the exposure printed above. This notebook does not create a COSI background PHA, so select no background unless one has been converted separately. XSPEC then creates the simulated COSI spectrum.

For an XMM--Newton forecast, use `xmm_sim_rmf` and `xmm_sim_arf`. For a NuSTAR forecast, use `nustar_sim_rmf` and `nustar_sim_arf`. The selected NuSTAR ARF corresponds to a 60-arcsec extraction radius at a 1-arcmin off-axis angle. These instructions produce source-only simulations with no input PHA or background file. Background can be added later if a realistic sensitivity forecast is required.

## 7B. Files to enter at the XSPEC `fakeit` prompts

Start each simulation with `data none`, define the desired model and parameters, and run `fakeit`. Enter the RMF, ARF, output filename, and exposure when XSPEC requests them. Use counting statistics so the output contains a Poisson realization rather than only the model expectation.

For XMM--Newton, use the XMM RMF and ARF and select the 0.3--10 keV band after creating the spectrum. For NuSTAR, use the NuSTAR RMF and matching ARF. For COSI, use the ARF and RMF written by this notebook. Choose no background for each `fakeit` run. The generated `.fak` files—not pre-existing PHA files—become the simulated spectra.

In [ ]:
fakeit_inputs = {
    "XMM RMF": xmm_sim_rmf,
    "XMM ARF": xmm_sim_arf,
    "NuSTAR RMF": nustar_sim_rmf,
    "NuSTAR ARF": nustar_sim_arf,
    "COSI RMF": rmf_path.resolve(),
    "COSI ARF": arf_path.resolve(),
}
for label, path in fakeit_inputs.items():
    print(f"{label:32s} {path}")

## Why `ftflx2xsp` is different

`ftflx2xsp` is useful when the input is a table of deconvolved flux points and one-sigma errors. It creates a unit diagonal response, so it does not retain the COSI effective area or redistribution. Use it for an approximate SED fit or visualization, not as a replacement for the ARF/RMF conversion above.